# Data Cleaning and Preparation for Half-Life Regression

The dataset was extracted from ENSDF (Evaluated Nuclear Structure Data File),
which contains evaluated nuclear structure and decay information for nuclides.

## Workflow
1. Load and inspect the ENSDF-derived dataset
2. Check duplicate records
3. Select nuclides with finite numerical half-life values
4. Convert half-life to log10(seconds) because half-life spans many orders of magnitude
5. Check missing values in candidate input features
6. Identify complete cases for the current candidate features
7. Identify redundant or unsuitable variables
8. Define candidate input features for regression
9. Prepare cleaned datasets for EDA and modeling

In [ ]:
#1. Load and inspect the ENSDF-derived dataset

import pandas as pd
import numpy as np

df = pd.read_csv("ensdf_ground_state_candidates.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (3371, 20)


,nuclide,element,Z,A,N,dataset_name,ground_energy_keV,spin_parity_raw,half_life_raw,half_life_value,half_life_unit,half_life_seconds,half_life_status,Q_beta_keV,S_n_keV,S_p_keV,Q_alpha_keV,ground_state_ambiguous,ground_state_candidate_count,source_file
0,1H,H,1,1,0,ADOPTED LEVELS,0.0,1/2+,STABLE,NaN,NaN,NaN,stable,NaN,NaN,NaN,NaN,False,1,ensdf.001
1,2H,H,1,2,1,ADOPTED LEVELS,0.0,1+,STABLE,NaN,NaN,NaN,stable,NaN,2224.566,2224.566,1.0,False,1,ensdf.002
2,3H,H,1,3,2,ADOPTED LEVELS,0.0,1/2+,12.32 Y,12.32,Y,388789632.0,finite,18.5906,6257.233,NaN,NaN,False,1,ensdf.003
3,3HE,HE,2,3,1,ADOPTED LEVELS,0.0,1/2+,STABLE,NaN,NaN,NaN,stable,NaN,NaN,5493.477,NaN,False,1,ensdf.003
4,4H,H,1,4,3,ADOPTED LEVELS,0.0,2-,NaN,NaN,NaN,NaN,missing,NaN,NaN,NaN,NaN,False,1,ensdf.004


In [7]:
#2. Check duplicate records

print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate nuclides:", df["nuclide"].duplicated().sum())

#Exact duplicate rows: Check whether any rows are completely identical across all columns.
#Duplicate nuclides: Check whether the same nuclide appears more than once in the dataset.

Exact duplicate rows: 0
Duplicate nuclides: 0


In [ ]:
# 3. Select nuclides with finite numerical half-life values
# For half-life regression, only nuclides with a finite numerical half-life can be used as training targets. 
# Stable, missing, or unparsed half-life records are excluded from the regression dataset.

reg_df = df[df["half_life_status"] == "finite"].copy()

print("Nuclides with finite half-life:", len(reg_df))

Nuclides with finite half-life: 2796


In [10]:
# 4. Convert half-life to log10(seconds)
# Why:
# Half-life values span many orders of magnitude 10^12 to 10^-12.
# Using log10 reduces the extreme numerical range and prevents very large
# half-life values from dominating the regression.
# Before the transformation, verify that all selected half-life values are
# numerical, non-missing, and positive.

print("Missing half-life values:",
      reg_df["half_life_seconds"].isna().sum())

print("Zero or negative half-life values:",
      (reg_df["half_life_seconds"] <= 0).sum())

reg_df["log_half_life"] = np.log10(reg_df["half_life_seconds"])

display(
    reg_df[
        ["nuclide", "half_life_seconds", "log_half_life"]
    ].head()
)

Missing half-life values: 0
Zero or negative half-life values: 0


,nuclide,half_life_seconds,log_half_life
2,3H,3.887896e+08,8.589715
11,6HE,8.067000e-01,-0.093288
17,7BE,4.598208e+06,6.662589
19,8HE,1.191000e-01,-0.924088
20,8LI,8.399000e-01,-0.075772


In [11]:

# 5. Check missing values in candidate input features
# Why:
# Missing values determine how many nuclides can be used for regression
# and help us decide whether to remove samples, reduce features, or use imputation.
#
# Note:
# spin_parity_raw is not included at this stage because it is stored as categorical/text information.
# It would require additional preprocessing/encoding before it can be used as a numerical model input.
# Whether to include spin/parity as an additional feature should be discussed.

candidate_features = [
    "Z",
    "N",
    "Q_beta_keV",
    "S_n_keV",
    "S_p_keV",
    "Q_alpha_keV"
]

missing_count = reg_df[candidate_features].isna().sum()
missing_percent = reg_df[candidate_features].isna().mean() * 100

missing_summary = pd.DataFrame({
    "Missing count": missing_count,
    "Missing (%)": missing_percent
})

display(missing_summary)

,Missing count,Missing (%)
Z,0,0.000000
N,0,0.000000
Q_beta_keV,826,29.542203
S_n_keV,624,22.317597
S_p_keV,630,22.532189
Q_alpha_keV,615,21.995708


In [12]:
# 6. Identify complete cases for the current candidate features
# Why:
# Missing values can overlap across different features.
# Therefore, we need to determine how many nuclides contain all six
# candidate numerical features before deciding how to handle missing data.
# No rows are permanently removed at this stage.

complete_df = reg_df.dropna(subset=candidate_features).copy()

print("Total finite half-life samples:", len(reg_df))
print("Complete cases:", len(complete_df))
print("Samples with at least one missing feature:",
      len(reg_df) - len(complete_df))
print("Retained percentage:",
      len(complete_df) / len(reg_df) * 100)

Total finite half-life samples: 2796
Complete cases: 1671
Samples with at least one missing feature: 1125
Retained percentage: 59.76394849785408


In [13]:
# 7. Identify redundant or unsuitable variables
# Why:
# Some variables may be redundant, constant, non-informative,
# or directly related to the target variable.
# These variables should be identified before defining the final regression inputs.

# Check columns that contain only one unique value
constant_columns = [
    col for col in reg_df.columns
    if reg_df[col].nunique(dropna=False) == 1
]

print("Constant columns:", constant_columns)

# Check whether mass number A is fully determined by Z and N
print(
    "A = Z + N for all rows:",
    (reg_df["A"] == reg_df["Z"] + reg_df["N"]).all()
)

# Identify columns that are directly derived from half-life information
# and therefore should not be used as model inputs to avoid data leakage.
target_related_columns = [
    "half_life_raw",
    "half_life_value",
    "half_life_unit",
    "half_life_seconds",
    "half_life_status"
]

print("Target-related columns to exclude:", target_related_columns)

Constant columns: ['ground_energy_keV', 'half_life_status', 'ground_state_ambiguous', 'ground_state_candidate_count']
A = Z + N for all rows: True
Target-related columns to exclude: ['half_life_raw', 'half_life_value', 'half_life_unit', 'half_life_seconds', 'half_life_status']


In [ ]:
# 8. Define candidate input features for regression
# Why:
# Based on the previous checks, we now define the initial numerical features
# to be used as model inputs for half-life regression.
#
# A is excluded because it is redundant with Z and N (A = Z + N).
# Half-life-related columns are excluded to avoid data leakage.
# Identifier/metadata columns are not used as predictive inputs.
# spin_parity_raw is temporarily excluded because it requires additional
# preprocessing/encoding; its inclusion should be discussed.

features = [
    "Z",
    "N",
    "Q_beta_keV",
    "S_n_keV",
    "S_p_keV",
    "Q_alpha_keV"
]

target = "log_half_life"

print("Regression input features:", features)
print("Regression target:", target)

Regression input features: ['Z', 'N', 'Q_beta_keV', 'S_n_keV', 'S_p_keV', 'Q_alpha_keV']
Regression target: log_half_life


In [15]:
# 9. Prepare cleaned datasets for EDA and modeling
# Why:
# EDA retains all nuclides with finite half-life values so that overall
# distributions, relationships, and missing-data patterns can be examined.
#
# The initial modeling dataset contains only complete cases for the six
# candidate numerical features.
# This complete-case dataset is only a baseline and may be revised after EDA
# depending on missing-data patterns and feature relationships.

# Dataset for exploratory data analysis
eda_df = reg_df[
    ["nuclide", "half_life_seconds", "log_half_life"] + features
].copy()

# Initial complete-case dataset for regression modeling
model_df = reg_df.dropna(subset=features)[
    features + [target]
].copy()

print("EDA dataset shape:", eda_df.shape)
print("Initial modeling dataset shape:", model_df.shape)

display(eda_df.head())
display(model_df.head())

EDA dataset shape: (2796, 9)
Initial modeling dataset shape: (1671, 7)


,nuclide,half_life_seconds,log_half_life,Z,N,Q_beta_keV,S_n_keV,S_p_keV,Q_alpha_keV
2,3H,3.887896e+08,8.589715,1,2,18.5906,6257.233,NaN,NaN
11,6HE,8.067000e-01,-0.093288,2,4,3505.2200,1710.000,NaN,NaN
17,7BE,4.598208e+06,6.662589,4,3,NaN,10677.000,5606.85,-1587.13
19,8HE,1.191000e-01,-0.924088,2,6,10663.8800,2535.000,NaN,NaN
20,8LI,8.399000e-01,-0.075772,3,5,16004.1300,2032.620,12416.00,NaN


,Z,N,Q_beta_keV,S_n_keV,S_p_keV,Q_alpha_keV,log_half_life
36,4,7,11509.3,501.64,20164.0,-8321.0,1.138618
42,4,8,11708.4,3170.70,22939.5,-8956.8,-1.668370
43,5,7,13369.4,3369.60,14096.7,-10001.3,-1.694649
45,7,5,-14577.0,15040.00,600.3,-8008.4,-1.958607
49,5,8,13436.9,4878.80,15804.8,-10817.9,-1.761954
